# 13 - Scoring Komentar + Agregasi Skor Influencer (C1)

Implementasi/pemakaian model IndoBERTweet hasil tahap 12: prediksi sentimen untuk SEMUA komentar
hasil scraping (bukan cuma sample yang dipakai training), lalu agregasi jadi skor sentimen (C1)
per influencer -- salah satu kriteria yang akan dipakai Entropy Weight + EDAS di tahap pemeringkatan.

**PENTING -- soal model yang dipakai:** checkpoint yang dipakai di sini
(`03_Indobert/models/<skema>/final/checkpoints/`) dilatih 2026-11-09, yaitu SEBELUM perbaikan
anti-leakage di tahap 12 final_scripts (split validasi internal yang HARUS cuma dari baris asli --
diperbaiki 2026-11-10). Jadi skor di bawah ini pakai model dari logic split yang LAMA (bocor
sedikit). Hasil `komentar_discoring.csv` & `skor_influencer.csv` di `output/13_scoring_komentar/`
adalah hasil nyata dari model itu (sudah dijalankan 2026-11-14) -- disalin apa adanya karena
proses scoring 22rb komentar + load 2 model besar tidak murah untuk diulang. **Kalau butuh angka
yang benar-benar bebas dari isu leakage di atas, re-train tahap 12 dengan logic yang sudah
diperbaiki dulu, baru jalankan ulang tahap ini.**

**Alur (1 cell per tahap):**
- **Blok A** -- Setup, config, load `comments.csv`, exclude `drrichardlee`.
- **Blok B** -- Bersihin teks komentar (cleaning, sama persis seperti tahap 07).
- **Blok C** -- Load model (3class & binary) dari checkpoint tahap 12, prediksi label tiap komentar.
- **Blok D** -- Hitung S_k per komentar (4 skenario) & agregasi jadi skor C1 per influencer.
- **Blok E** -- [Opsional, eksploratif] cek manual sampel komentar negatif/positif per influencer,
  distribusi label, kuantifikasi pengaruh emoji ke prediksi -- cuma baca ulang hasil, tidak nulis
  data baru.
- **Blok F** -- Bandingkan C1 dari semua skenario sekaligus (korelasi rank Spearman) & rekap label
  per influencer.

**Konsolidasi dari:** `03_Indobert/03_labeling_all_data.ipynb`

**Catatan eksekusi:** Blok C butuh GPU + 2 model checkpoint (~1.3GB masing-masing) + akses ke
`comments.csv` (22rb+ baris) -- berat. Skip otomatis kalau `komentar_discoring.csv` &
`skor_influencer.csv` sudah ada.

In [ ]:
import re
import gc
from pathlib import Path

import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# KONFIGURASI PATH
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "13_scoring_komentar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path.cwd().parent  # final_scripts/ ada tepat di bawah root repo

PATH_COMMENTS = FINAL_OUTPUT_ROOT / "06_scraping_komentar" / "comments.csv"
# Model checkpoint TETAP di lokasi lama -- ~1.3GB per model, sengaja TIDAK disalin ke output/
# (sama seperti cookies TikTok di tahap 06: file besar/kredensial, direferensikan di tempat).
MODELS_DIR = REPO_ROOT / "03_Indobert" / "models"

OUT_KOMENTAR_DISCORING = OUTPUT_DIR / "komentar_discoring.csv"
OUT_SKOR_INFLUENCER = OUTPUT_DIR / "skor_influencer.csv"

# --- Username yang DIKECUALIKAN, ga dilanjut ke tahap ini ---
EXCLUDE_USERNAMES = ["drrichardlee"]

# --- Aturan cleaning (sama kaya tahap 07) ---
MIN_WORDS = 2

# --- Model ---
MODEL_NAME = "indolem/indobertweet-base-uncased"
MAX_LEN = 56           # sama kaya training (comment-only, ga pake caption)
BATCH_SIZE = 64

# --- Mapping label -> index (harus SAMA PERSIS kaya pas training di tahap 12) ---
LABEL2ID = {
    "binary": {"non-negatif": 0, "negatif": 1},
    "3class": {"negatif": 0, "netral": 1, "positif": 2},
}

# --- Mapping label -> S_k (skor sentimen per komentar) ---
# 3class        : positif=+1, netral=0, negatif=-1
# 3class_hanya_positif: cuma komentar POSITIF yang nambah skor, netral & negatif dianggap 0
# binary varianA: cuma komentar NEGATIF yang "menghukum" skor, non-negatif dianggap netral (0)
# binary varianB: simetris, negatif=-1 non-negatif=+1 (kaya 3class versi 2 kelas)
SK_MAP = {
    "3class": {"negatif": -1, "netral": 0, "positif": 1},
    "3class_hanya_positif": {"negatif": 0, "netral": 0, "positif": 1},
    "binary_varianA": {"negatif": -1, "non-negatif": 0},
    "binary_varianB": {"negatif": -1, "non-negatif": 1},
}

print("Config siap. EXCLUDE_USERNAMES =", EXCLUDE_USERNAMES)

## BLOK A - Load comments.csv + exclude drrichardlee

In [ ]:
def _find_latest_checkpoint(scheme: str) -> Path:
    """Ambil folder checkpoint-N dengan N terbesar di models/<skema>/final/checkpoints/."""
    ckpt_dir = MODELS_DIR / scheme / "final" / "checkpoints"
    if not ckpt_dir.exists():
        raise FileNotFoundError(f"{ckpt_dir} tidak ada. Jalankan tahap 12 (fine-tuning) dulu di mesin GPU.")
    candidates = sorted(ckpt_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    if not candidates:
        raise FileNotFoundError(f"Tidak ada folder checkpoint-* di {ckpt_dir}.")
    return candidates[-1]


if not PATH_COMMENTS.exists():
    raise FileNotFoundError(f"{PATH_COMMENTS} tidak ditemukan. Jalankan tahap 06 dulu.")

df_raw = pd.read_csv(PATH_COMMENTS, dtype=str, keep_default_na=False, encoding="utf-8")

required = {"username", "video_id", "commenter_username", "comment_text"}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f"comments.csv kurang kolom: {missing}")

n0 = len(df_raw)
df = df_raw[~df_raw["username"].isin(EXCLUDE_USERNAMES)].reset_index(drop=True)
print(f"Komentar mentah       : {n0:,}")
print(f"Setelah exclude {EXCLUDE_USERNAMES}: {len(df):,}  (dibuang {n0 - len(df):,})")

## BLOK B - Bersihin Teks Komentar

In [ ]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
MENTION_RE = re.compile(r"@[A-Za-z0-9_.]+")
SPACE_RE = re.compile(r"\s+")


def bersihkan_komentar(comments_df, min_words=MIN_WORDS):
    """Langkah2 PERSIS kaya tahap 07: buang kosong -> hapus URL/mention -> dedup
    (video+akun+teks sama) -> buang tanpa huruf -> buang <min_words kata. Emoji dibiarin,
    ga ada stemming/stopword/dsb."""
    df = comments_df.copy()
    df["comment_text_raw"] = df["comment_text"]
    awal = len(df)

    def catat(langkah, sebelum, sesudah):
        print(f"  {langkah}  ({sebelum:,} -> {sesudah:,}, dibuang {sebelum - sesudah:,})")

    n0 = len(df)
    df = df[df["comment_text"].str.strip() != ""]
    catat("1. Komentar kosong", n0, len(df))

    n0 = len(df)
    df = df.assign(comment_text=(
        df["comment_text"]
        .str.replace(URL_RE, " ", regex=True)
        .str.replace(MENTION_RE, " ", regex=True)
        .str.replace(SPACE_RE, " ", regex=True)
        .str.strip()
    ))
    df = df[df["comment_text"] != ""]
    catat("2. Hapus URL/mention, jadi kosong", n0, len(df))

    n0 = len(df)
    df = df.drop_duplicates(subset=["video_id", "commenter_username", "comment_text"], keep="first")
    catat("3. Duplikat (video+akun+teks sama)", n0, len(df))

    n0 = len(df)
    df = df[df["comment_text"].str.contains(r"[A-Za-z]", regex=True)]
    catat("4. Tanpa huruf (emoji/tanda baca saja)", n0, len(df))

    n0 = len(df)
    df = df[df["comment_text"].str.split().str.len() >= min_words]
    catat(f"5. Kurang dari {min_words} kata", n0, len(df))

    df = df.rename(columns={"comment_text": "comment_text_clean"}).reset_index(drop=True)
    print(f"  TOTAL: {awal:,} mentah -> {len(df):,} bersih")
    return df


print("Pembersihan komentar:")
df = bersihkan_komentar(df)

## BLOK C - Load Model & Prediksi

Butuh GPU + model-nya ada (hasil tahap 12). Skip kalau `komentar_discoring.csv` sudah ada.

In [ ]:
def load_model_dan_tokenizer(checkpoint_path, model_name=MODEL_NAME):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(str(checkpoint_path))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return model, tokenizer, device


def prediksi_batch(texts, model, tokenizer, device, max_len=MAX_LEN, batch_size=BATCH_SIZE):
    """Prediksi index label (int) buat list teks, batch demi batch biar ga OOM."""
    import torch

    semua_pred = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(
                list(batch), truncation=True, padding="max_length",
                max_length=max_len, return_tensors="pt",
            ).to(device)
            logits = model(**enc).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            semua_pred.extend(preds.tolist())
            if (i // batch_size) % 20 == 0:
                print(f"    {i + len(batch):,}/{len(texts):,} komentar diprediksi...")
    return semua_pred


print("Fungsi load_model_dan_tokenizer() & prediksi_batch() siap.")

### Prediksi model 3CLASS

In [ ]:
if OUT_KOMENTAR_DISCORING.exists():
    print(f"[Prediksi 3class] {OUT_KOMENTAR_DISCORING} sudah ada -> skip (hapus file ini kalau mau re-score).")
else:
    import torch

    id2label_3class = {v: k for k, v in LABEL2ID["3class"].items()}
    ckpt_3class = _find_latest_checkpoint("3class")
    print("Load model 3class dari", ckpt_3class)
    model_3c, tok_3c, device = load_model_dan_tokenizer(ckpt_3class)
    print("  device:", device)

    print("Prediksi (3class)...")
    pred_idx_3c = prediksi_batch(df["comment_text_clean"].tolist(), model_3c, tok_3c, device)
    df["pred_label_3class"] = [id2label_3class[i] for i in pred_idx_3c]

    del model_3c
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    print(df["pred_label_3class"].value_counts())

### Prediksi model BINARY

In [ ]:
if OUT_KOMENTAR_DISCORING.exists():
    print(f"[Prediksi binary] {OUT_KOMENTAR_DISCORING} sudah ada -> skip (hapus file ini kalau mau re-score).")
else:
    import torch

    id2label_binary = {v: k for k, v in LABEL2ID["binary"].items()}
    ckpt_binary = _find_latest_checkpoint("binary")
    print("Load model binary dari", ckpt_binary)
    model_bin, tok_bin, device = load_model_dan_tokenizer(ckpt_binary)
    print("  device:", device)

    print("Prediksi (binary)...")
    pred_idx_bin = prediksi_batch(df["comment_text_clean"].tolist(), model_bin, tok_bin, device)
    df["pred_label_binary"] = [id2label_binary[i] for i in pred_idx_bin]

    del model_bin
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    print(df["pred_label_binary"].value_counts())

## BLOK D - Hitung S_k & Agregasi Skor Influencer (C1)

In [ ]:
def hitung_sk(pred_labels, sk_map):
    """pred_labels = array nama label (str) hasil prediksi model. Return array S_k (int)."""
    tidak_dikenal = set(pred_labels) - set(sk_map.keys())
    if tidak_dikenal:
        raise ValueError(f"Ada label prediksi yg ga ada di sk_map: {tidak_dikenal}")
    return np.array([sk_map[lbl] for lbl in pred_labels])


if OUT_KOMENTAR_DISCORING.exists():
    print(f"[S_k] {OUT_KOMENTAR_DISCORING} sudah ada -> skip hitung ulang.")
else:
    df["sk_3class"] = hitung_sk(df["pred_label_3class"], SK_MAP["3class"])
    df["sk_3class_hanya_positif"] = hitung_sk(df["pred_label_3class"], SK_MAP["3class_hanya_positif"])
    df["sk_binary_varianA"] = hitung_sk(df["pred_label_binary"], SK_MAP["binary_varianA"])
    df["sk_binary_varianB"] = hitung_sk(df["pred_label_binary"], SK_MAP["binary_varianB"])

    kolom_simpan = ["username", "video_id", "commenter_username", "comment_text_raw",
                     "comment_text_clean", "pred_label_3class", "pred_label_binary",
                     "sk_3class", "sk_binary_varianA", "sk_binary_varianB"]
    df[kolom_simpan].to_csv(OUT_KOMENTAR_DISCORING, index=False, encoding="utf-8")
    print(f"Tersimpan: {OUT_KOMENTAR_DISCORING}  ({len(df):,} baris)")

### Agregasi C1 per influencer (rata-rata S_k, semua skenario)

In [ ]:
def agregasi_c1(df, kolom_sk, kolom_influencer="username"):
    """C1 = rata-rata S_k per influencer."""
    agg = df.groupby(kolom_influencer)[kolom_sk].agg(
        C1="mean", P="count", total_Sk="sum",
    ).reset_index()
    return agg


def gabung_c1_semua_skema(df):
    hasil = None
    for nama_skema, kolom_sk in [
        ("3class", "sk_3class"),
        ("3class_hanya_positif", "sk_3class_hanya_positif"),
        ("binary_varianA", "sk_binary_varianA"),
        ("binary_varianB", "sk_binary_varianB"),
    ]:
        agg = agregasi_c1(df, kolom_sk)
        agg = agg.rename(columns={
            "C1": f"C1_{nama_skema}", "P": f"P_{nama_skema}", "total_Sk": f"total_Sk_{nama_skema}",
        })
        hasil = agg if hasil is None else hasil.merge(agg, on="username", how="outer")
    return hasil


if OUT_SKOR_INFLUENCER.exists():
    print(f"[C1] {OUT_SKOR_INFLUENCER} sudah ada -> skip.")
else:
    skor_influencer = gabung_c1_semua_skema(df)
    skor_influencer = skor_influencer.sort_values("C1_3class", ascending=False).reset_index(drop=True)
    skor_influencer.to_csv(OUT_SKOR_INFLUENCER, index=False, encoding="utf-8")
    print(f"Tersimpan: {OUT_SKOR_INFLUENCER}  ({len(skor_influencer):,} influencer)")

skor_influencer = pd.read_csv(OUT_SKOR_INFLUENCER)
print("\nPreview (urut C1_3class tertinggi):")
print(skor_influencer.head(10).to_string())

## BLOK E - Cek Manual Sampel Komentar Negatif/Positif per Influencer

Buat validasi: ngecek apa prediksi model emang masuk akal buat influencer yang hasil C1-nya
keliatan ga sesuai ekspektasi. Ini bukan nambah data baru, cuma baca ulang hasil yang udah
disimpen terus nampilin beberapa sampel komentar buat dibaca manual.

In [ ]:
df_cek = pd.read_csv(OUT_KOMENTAR_DISCORING, dtype=str, keep_default_na=False, encoding="utf-8")
print(f"Loaded {len(df_cek):,} komentar dari {OUT_KOMENTAR_DISCORING.name}")


def tampilkan_sampel_label(df, usernames, label, kolom_pred="pred_label_3class", n_sampel=20, seed=42):
    """skema label = negatif/netral/positif (3class) atau negatif/non-negatif (binary).
    Ambil sampel RANDOM (bukan yg pertama doang) biar representatif."""
    hasil = {}
    for u in usernames:
        sub = df[(df["username"] == u) & (df[kolom_pred] == label)]
        n_ambil = min(n_sampel, len(sub))
        sampel = sub.sample(n=n_ambil, random_state=seed) if n_ambil > 0 else sub
        hasil[u] = sampel
        print(f"\n=== {u} -- {len(sub)} komentar '{label}' ({kolom_pred}), nampilin {n_ambil} sampel ===")
        for _, row in sampel.iterrows():
            print(f"  - {row['comment_text_clean']}")
    return hasil


print("Fungsi tampilkan_sampel_label() siap.")

### Contoh: cek komentar negatif (3class) untuk 2 influencer

In [ ]:
_ = tampilkan_sampel_label(df_cek, ["tirtacipeng", "giapratamamd"], label="negatif", kolom_pred="pred_label_3class", n_sampel=20)

### Distribusi jumlah & persentase label (3class & binary), seluruh data

In [ ]:
total = len(df_cek)
print(f"Total komentar yang discoring: {total:,}\n")

print("-- Distribusi pred_label_3class --")
for label, jumlah in df_cek["pred_label_3class"].value_counts().items():
    pct = 100 * jumlah / total
    print(f"  {label:10s}: {jumlah:6,}  ({pct:5.1f}%)")

print("\n-- Distribusi pred_label_binary --")
for label, jumlah in df_cek["pred_label_binary"].value_counts().items():
    pct = 100 * jumlah / total
    print(f"  {label:14s}: {jumlah:6,}  ({pct:5.1f}%)")

### Kuantifikasi -- seberapa kuat 1 emoji tertentu nyeret prediksi ke "negatif"

In [ ]:
def analisis_emoji_vs_label(df, emoji="\U0001F62D", kolom_pred="pred_label_3class"):
    df = df.copy()
    df["ada_emoji"] = df["comment_text_clean"].str.contains(emoji, regex=False)

    print(f"Proporsi komentar yang ngandung '{emoji}', per label prediksi ({kolom_pred}):")
    tabel = df.groupby(kolom_pred)["ada_emoji"].agg(jumlah_ada_emoji="sum", total="count")
    tabel["persen"] = 100 * tabel["jumlah_ada_emoji"] / tabel["total"]
    print(tabel.to_string())

    total_emoji = df["ada_emoji"].sum()
    negatif_dgn_emoji = ((df[kolom_pred] == "negatif") & df["ada_emoji"]).sum()
    total_negatif = (df[kolom_pred] == "negatif").sum()
    print(f"\nDari {total_negatif} komentar berlabel negatif, {negatif_dgn_emoji} "
          f"({100*negatif_dgn_emoji/total_negatif:.1f}%) ngandung '{emoji}'.")
    print(f"Dari {total_emoji} komentar yang ngandung '{emoji}' total, "
          f"{negatif_dgn_emoji} ({100*negatif_dgn_emoji/total_emoji:.1f}%) dilabel negatif.")
    return tabel


_ = analisis_emoji_vs_label(df_cek, emoji="\U0001F62D", kolom_pred="pred_label_3class")  # emoji wajah menangis keras

## BLOK F - Bandingin Semua Skenario Skor & Rekap per Influencer

In [ ]:
def bandingkan_semua_c1(skor_df, kolom_c1=None):
    """Tabel sisi-berdampingan + korelasi rank (Spearman) antar skenario S_k -- buat liat
    seberapa beda rangking influencer kalau skema skoringnya diganti."""
    if kolom_c1 is None:
        kolom_c1 = [c for c in skor_df.columns if c.startswith("C1_")]

    tabel = skor_df[["username"] + kolom_c1].sort_values(kolom_c1[0], ascending=False).reset_index(drop=True)
    print(f"Tabel perbandingan C1 per skenario (urut berdasarkan {kolom_c1[0]}):")
    print(tabel.to_string())

    print("\nKorelasi rank (Spearman) antar skenario -- makin deket 1.0 berarti rangkingnya makin mirip:")
    corr = skor_df[kolom_c1].corr(method="spearman")
    print(corr.round(3).to_string())
    return tabel, corr


tabel_c1, korelasi_c1 = bandingkan_semua_c1(skor_influencer)

### Rekap jumlah negatif/netral/positif per dokter

In [ ]:
def rekap_label_per_influencer(df, kolom_pred="pred_label_3class", usernames=None):
    """Tabel jumlah & persen komentar per label (negatif/netral/positif), per influencer."""
    sub = df if usernames is None else df[df["username"].isin(usernames)]
    rekap = sub.groupby(["username", kolom_pred]).size().unstack(fill_value=0)
    rekap["total"] = rekap.sum(axis=1)
    for kolom in rekap.columns:
        if kolom != "total":
            rekap[f"{kolom}_pct"] = (100 * rekap[kolom] / rekap["total"]).round(1)
    return rekap


print("Rekap jumlah & persen per label (3class) -- SEMUA dokter:")
rekap = rekap_label_per_influencer(df_cek, usernames=None)
print(rekap.to_string())